# Detection de Panneaux Routiers - Custom YOLO Notebook

Notebook Kaggle pour entrainer une implementation YOLO simple en TensorFlow/Keras sur le dataset clone depuis `https://github.com/OmarLabiade/ProjetAP`.

Avant execution sur Kaggle:
- activer Internet
- activer GPU


## 1. Imports et configuration


In [ ]:
from __future__ import annotations

import math
import os
import random
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import seaborn as sns
import tensorflow as tf
from PIL import Image
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Conv2D, Dropout, Input, MaxPooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_REPO_URL = "https://github.com/OmarLabiade/ProjetAP.git"
DATASET_CLONE_DIR = Path("/kaggle/working/ProjetAP_repo")
DATASET_SUBDIR = "dataset"
IMAGE_H = 384
IMAGE_W = 768
STRIDE = 16
GRID_H = IMAGE_H // STRIDE
GRID_W = IMAGE_W // STRIDE
NB_CLASSES = 6
PIX_PER_CELL_X = IMAGE_W // GRID_W
PIX_PER_CELL_Y = IMAGE_H // GRID_H

BATCH_SIZE = 2
EPOCHS = 75
LEARNING_RATE = 3e-4
LAMBDA_COORD = 5.0
LAMBDA_NOOBJ = 0.5
CONFIDENCE_THRESHOLD = 0.20
IOU_THRESHOLD = 0.50

CLASS_LABELS = ["complement", "danger", "direction", "indication", "localisation", "ordre"]
COLORS = ["cyan", "red", "blue", "green", "orange", "magenta"]

RUN_DIR = Path("/kaggle/working/custom_yolo_run")
RUN_DIR.mkdir(parents=True, exist_ok=True)
BEST_WEIGHTS_PATH = RUN_DIR / "best.weights.h5"

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 2. Recuperation des donnees


In [ ]:
def run(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


def resolve_dataset_dir(root: Path) -> Path:
    if (root / "train").is_dir() and (root / "test").is_dir() and ((root / "val").is_dir() or (root / "valid").is_dir()):
        return root
    nested = root / DATASET_SUBDIR
    if (nested / "train").is_dir() and (nested / "test").is_dir() and ((nested / "val").is_dir() or (nested / "valid").is_dir()):
        return nested
    raise FileNotFoundError(f"Could not find train/test/val(or valid) under {root}")


def clone_dataset_repo(repo_url: str, clone_dir: Path) -> Path:
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    run(["git", "clone", "--depth", "1", repo_url, str(clone_dir)])
    return clone_dir


def get_split_dirs(dataset_root: Path) -> tuple[Path, Path, Path]:
    train_dir = dataset_root / "train"
    val_dir = dataset_root / "val" if (dataset_root / "val").is_dir() else dataset_root / "valid"
    test_dir = dataset_root / "test"
    return train_dir, val_dir, test_dir


def count_images(folder: Path) -> int:
    count = 0
    for ext in ("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"):
        count += len(list(folder.glob(ext)))
    return count


repo_root = clone_dataset_repo(DATASET_REPO_URL, DATASET_CLONE_DIR)
dataset_root = resolve_dataset_dir(repo_root)
PATH_TRAIN, PATH_VALID, PATH_TEST = get_split_dirs(dataset_root)

print("Dataset root:", dataset_root)
print("Train:", count_images(PATH_TRAIN), "images")
print("Valid:", count_images(PATH_VALID), "images")
print("Test :", count_images(PATH_TEST), "images")
print(f"Grid : {GRID_H}x{GRID_W}, sortie reseau : ({GRID_H}, {GRID_W}, {1 + 4 + NB_CLASSES})")


## 3. Chargement des donnees


In [ ]:
def find_image_for_stem(stem_path: Path) -> Path | None:
    stem_str = str(stem_path)
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
        candidate = Path(stem_str + ext)
        if candidate.exists():
            return candidate
    return None


def load_data(ds_path: Path):
    txt_files = sorted(ds_path.glob("*.txt"))
    pairs = []

    for txt_path in txt_files:
        image_path = find_image_for_stem(txt_path.with_suffix(""))
        if image_path is not None:
            pairs.append((image_path, txt_path))

    print(f"  {len(pairs)} images dans {ds_path}")

    x = np.zeros((len(pairs), IMAGE_H, IMAGE_W, 3), dtype=np.uint8)
    y = []

    for i, (img_path, txt_path) in enumerate(pairs):
        img = Image.open(img_path).convert("RGB")
        src_w, src_h = img.size
        scale = min(IMAGE_W / src_w, IMAGE_H / src_h)
        new_w = int(round(src_w * scale))
        new_h = int(round(src_h * scale))
        pad_x = (IMAGE_W - new_w) // 2
        pad_y = (IMAGE_H - new_h) // 2

        resized = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        canvas = Image.new("RGB", (IMAGE_W, IMAGE_H), (114, 114, 114))
        canvas.paste(resized, (pad_x, pad_y))
        x[i] = np.array(canvas, dtype=np.uint8)

        boxes = []
        for line in txt_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls = int(parts[0])
            cx, cy, w, h = map(float, parts[1:])

            x1 = (cx - w / 2.0) * src_w * scale + pad_x
            y1 = (cy - h / 2.0) * src_h * scale + pad_y
            x2 = (cx + w / 2.0) * src_w * scale + pad_x
            y2 = (cy + h / 2.0) * src_h * scale + pad_y

            x1 = np.clip(x1, 0, IMAGE_W - 1)
            y1 = np.clip(y1, 0, IMAGE_H - 1)
            x2 = np.clip(x2, 0, IMAGE_W - 1)
            y2 = np.clip(y2, 0, IMAGE_H - 1)

            bw = max((x2 - x1) / IMAGE_W, 1.0 / IMAGE_W)
            bh = max((y2 - y1) / IMAGE_H, 1.0 / IMAGE_H)
            bcx = ((x1 + x2) / 2.0) / IMAGE_W
            bcy = ((y1 + y2) / 2.0) / IMAGE_H
            boxes.append([bcx, bcy, bw, bh, cls])

        y.append(boxes)

    return x, y


print("Chargement...")
x_train_raw, y_train_raw = load_data(PATH_TRAIN)
x_val_raw, y_val_raw = load_data(PATH_VALID)
x_test_raw, y_test_raw = load_data(PATH_TEST)
print(f"Train={x_train_raw.shape[0]}, Valid={x_val_raw.shape[0]}, Test={x_test_raw.shape[0]}")


## 4. Statistiques de la base


In [ ]:
def stats_dataset(y, name):
    counts = [0] * NB_CLASSES
    for boxes in y:
        for b in boxes:
            counts[int(b[4])] += 1
    total = sum(counts)
    print(f"\n=== {name} - {len(y)} images, {total} objets ===")
    for i, label in enumerate(CLASS_LABELS):
        print(f"  {label:15s}: {counts[i]:4d} ({100 * counts[i] / max(total, 1):.1f}%)")
    plt.figure(figsize=(8, 3))
    plt.bar(CLASS_LABELS, counts, color=COLORS)
    plt.title(f"Distribution - {name}")
    plt.ylabel("Instances")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


stats_dataset(y_train_raw, "Train")
stats_dataset(y_val_raw, "Validation")
stats_dataset(y_test_raw, "Test")


## 5. Visualisation des donnees


In [ ]:
def afficher_image(x, y, idx=None):
    if idx is None:
        idx = np.random.randint(len(x))
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(x[idx] / 255.0)
    for b in y[idx]:
        cx, cy, w, h, cid = b[0], b[1], b[2], b[3], int(b[4])
        rect = patches.Rectangle(
            ((cx - w / 2) * IMAGE_W, (cy - h / 2) * IMAGE_H),
            w * IMAGE_W,
            h * IMAGE_H,
            linewidth=2,
            edgecolor=COLORS[cid],
            facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(
            (cx - w / 2) * IMAGE_W,
            max((cy - h / 2) * IMAGE_H - 3, 0),
            CLASS_LABELS[cid],
            color=COLORS[cid],
            fontsize=8,
            fontweight="bold",
            bbox=dict(facecolor="black", alpha=0.4, pad=1),
        )
    ax.set_title(f"Image {idx} - {len(y[idx])} panneau(x)")
    ax.axis("off")
    plt.show()


for i in range(min(4, len(x_train_raw))):
    afficher_image(x_train_raw, y_train_raw, idx=i)


## 6. Conversion au format tenseur YOLO


In [ ]:
def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def softmax_np(x):
    x = np.asarray(x, dtype=np.float32)
    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)


def set_box_for_yolo(y):
    y_yolo = np.zeros((len(y), GRID_H, GRID_W, 1 + 4 + NB_CLASSES), dtype=np.float32)
    for i, boxes in enumerate(y):
        boxes = sorted(boxes, key=lambda b: b[2] * b[3], reverse=True)
        for b in boxes:
            cx, cy, w, h, cid = b[0], b[1], b[2], b[3], int(b[4])
            cx_pix, cy_pix = cx * IMAGE_W, cy * IMAGE_H
            ix = min(int(cx * GRID_W), GRID_W - 1)
            iy = min(int(cy * GRID_H), GRID_H - 1)
            if y_yolo[i, iy, ix, 0] == 1:
                continue
            y_yolo[i, iy, ix, 0] = 1
            y_yolo[i, iy, ix, 1] = (cx_pix - ix * PIX_PER_CELL_X) / PIX_PER_CELL_X
            y_yolo[i, iy, ix, 2] = (cy_pix - iy * PIX_PER_CELL_Y) / PIX_PER_CELL_Y
            y_yolo[i, iy, ix, 3] = w
            y_yolo[i, iy, ix, 4] = h
            y_yolo[i, iy, ix, 5:] = to_categorical(cid, num_classes=NB_CLASSES)
    return y_yolo


def get_box_from_yolo(y_yolo, mode=None, confidence_threshold=0.1):
    result = []
    for i in range(len(y_yolo)):
        boxes = []
        for iy in range(GRID_H):
            for ix in range(GRID_W):
                cell = y_yolo[i, iy, ix]
                pres = float(sigmoid_np(cell[0])) if mode == "pred" else float(cell[0])
                if pres < confidence_threshold:
                    continue
                probs = softmax_np(cell[5:]) if mode == "pred" else cell[5:]
                cid = int(np.argmax(probs))
                cx_rel = float(sigmoid_np(cell[1])) if mode == "pred" else float(cell[1])
                cy_rel = float(sigmoid_np(cell[2])) if mode == "pred" else float(cell[2])
                w = float(sigmoid_np(cell[3]) ** 2) if mode == "pred" else float(cell[3])
                h = float(sigmoid_np(cell[4]) ** 2) if mode == "pred" else float(cell[4])
                cx = (cx_rel * PIX_PER_CELL_X + ix * PIX_PER_CELL_X) / IMAGE_W
                cy = (cy_rel * PIX_PER_CELL_Y + iy * PIX_PER_CELL_Y) / IMAGE_H
                cx = float(np.clip(cx, 0.0, 1.0))
                cy = float(np.clip(cy, 0.0, 1.0))
                w = float(np.clip(w, 1.0 / IMAGE_W, 1.0))
                h = float(np.clip(h, 1.0 / IMAGE_H, 1.0))
                boxes.append([cx, cy, w, h, cid, pres])
        result.append(boxes)
    return result


def cxcywh_to_xyxy(box):
    cx, cy, w, h = box[:4]
    return [cx - w / 2.0, cy - h / 2.0, cx + w / 2.0, cy + h / 2.0]


def apply_classwise_nms(boxes, iou_threshold=0.5):
    if len(boxes) == 0:
        return []

    final_boxes = []
    for cid in range(NB_CLASSES):
        cls_boxes = [b for b in boxes if int(b[4]) == cid]
        if not cls_boxes:
            continue
        cls_boxes = sorted(cls_boxes, key=lambda b: b[5], reverse=True)
        kept = []
        while cls_boxes:
            current = cls_boxes.pop(0)
            kept.append(current)
            current_xyxy = cxcywh_to_xyxy(current)
            remaining = []
            for candidate in cls_boxes:
                cand_xyxy = cxcywh_to_xyxy(candidate)
                x1 = max(current_xyxy[0], cand_xyxy[0])
                y1 = max(current_xyxy[1], cand_xyxy[1])
                x2 = min(current_xyxy[2], cand_xyxy[2])
                y2 = min(current_xyxy[3], cand_xyxy[3])
                inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
                area1 = max(0.0, current_xyxy[2] - current_xyxy[0]) * max(0.0, current_xyxy[3] - current_xyxy[1])
                area2 = max(0.0, cand_xyxy[2] - cand_xyxy[0]) * max(0.0, cand_xyxy[3] - cand_xyxy[1])
                union = area1 + area2 - inter
                iou_val = inter / union if union > 0 else 0.0
                if iou_val < iou_threshold:
                    remaining.append(candidate)
            cls_boxes = remaining
        final_boxes.extend(kept)

    return sorted(final_boxes, key=lambda b: b[5], reverse=True)


y_check = set_box_for_yolo(y_train_raw[:2])
print("Tenseur YOLO:", y_check.shape)
if len(y_train_raw) > 0 and len(y_train_raw[0]) > 0:
    print("Original    :", y_train_raw[0][0])
    print("Aller-retour:", get_box_from_yolo(y_check[:1])[0][0])


## 7. Preparation des tenseurs


In [ ]:
x_train = x_train_raw.astype(np.float32) / 255.0
x_val = x_val_raw.astype(np.float32) / 255.0
x_test = x_test_raw.astype(np.float32) / 255.0

y_train_YOLO = set_box_for_yolo(y_train_raw)
y_val_YOLO = set_box_for_yolo(y_val_raw)
y_test_YOLO = set_box_for_yolo(y_test_raw)

print("x_train     :", x_train.shape)
print("y_train_YOLO:", y_train_YOLO.shape)


## 8. Architecture du reseau


In [ ]:
def create_model_yolo(input_shape=(IMAGE_H, IMAGE_W, 3)):
    inp = Input(shape=input_shape)

    x = Conv2D(32, 3, padding="same", activation="elu", kernel_initializer="he_normal")(inp)
    x = BatchNormalization()(x)
    x = Conv2D(32, 3, padding="same", activation="elu", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2)(x)
    x = Dropout(0.1)(x)

    x = Conv2D(64, 3, padding="same", activation="elu", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    x = Conv2D(64, 3, padding="same", activation="elu", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2)(x)
    x = Dropout(0.15)(x)

    x = Conv2D(128, 3, padding="same", activation="elu", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    x = Conv2D(128, 3, padding="same", activation="elu", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2)(x)
    x = Dropout(0.2)(x)

    x = Conv2D(256, 3, padding="same", activation="elu", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    x = Conv2D(256, 3, padding="same", activation="elu", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D(2)(x)
    x = Dropout(0.2)(x)

    x = Conv2D(512, 3, padding="same", activation="elu", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)
    x = Conv2D(256, 3, padding="same", activation="elu", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)

    output = Conv2D(1 + 4 + NB_CLASSES, 1, padding="same", activation="linear", name="yolo_output")(x)
    return Model(inp, output)


model = create_model_yolo()
model.summary()
print(f"Sortie : {model.output_shape} - attendu : (None, {GRID_H}, {GRID_W}, {1 + 4 + NB_CLASSES})")


## 9. Fonction de cout YOLO


In [ ]:
def yolo_loss(lambda_coord=5.0, lambda_noobj=0.5):
    yolo_size = 1 + 4 + NB_CLASSES
    huber = tf.keras.losses.Huber(reduction=tf.keras.losses.Reduction.SUM)

    def loss_fn(y_true, y_pred):
        yt = tf.reshape(y_true, [-1, yolo_size])
        yp = tf.reshape(y_pred, [-1, yolo_size])
        obj_mask = yt[:, 0:1]

        pred_obj = yp[:, 0:1]
        pred_xy = tf.sigmoid(yp[:, 1:3])
        pred_wh = tf.square(tf.sigmoid(yp[:, 3:5]))
        pred_cls = yp[:, 5:]

        obj_loss = tf.nn.sigmoid_cross_entropy_with_logits(labels=yt[:, 0:1], logits=pred_obj)
        obj_loss = tf.reduce_sum(obj_mask * obj_loss)

        noobj_loss = tf.nn.sigmoid_cross_entropy_with_logits(labels=yt[:, 0:1], logits=pred_obj)
        noobj_loss = tf.reduce_sum((1.0 - obj_mask) * noobj_loss)

        coord_loss_xy = huber(yt[:, 1:3] * obj_mask, pred_xy * obj_mask)
        coord_loss_wh = huber(
            tf.sqrt(tf.maximum(yt[:, 3:5], 1e-6)) * obj_mask,
            tf.sqrt(tf.maximum(pred_wh, 1e-6)) * obj_mask,
        )

        class_loss = tf.nn.softmax_cross_entropy_with_logits(labels=yt[:, 5:], logits=pred_cls)
        class_loss = tf.reduce_sum(class_loss[:, None] * obj_mask)

        normalizer = tf.maximum(tf.reduce_sum(obj_mask), 1.0)
        total = (
            lambda_coord * (coord_loss_xy + coord_loss_wh) / normalizer
            + class_loss / normalizer
            + obj_loss / normalizer
            + lambda_noobj * noobj_loss / tf.cast(tf.shape(yt)[0], tf.float32)
        )
        return total

    return loss_fn


print("Loss YOLO definie.")


## 10. Entrainement


In [ ]:
model = create_model_yolo()
loss_fn = yolo_loss(lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ)
model.compile(loss=loss_fn, optimizer=Adam(learning_rate=LEARNING_RATE, clipnorm=5.0))

callbacks = [
    ModelCheckpoint(str(BEST_WEIGHTS_PATH), monitor="val_loss", save_weights_only=True, save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True, verbose=1),
]

history = model.fit(
    x_train,
    y_train_YOLO,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_val, y_val_YOLO),
    callbacks=callbacks,
    verbose=1,
)


## 11. Courbes d'apprentissage


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history["loss"], "b--", label="Train")
plt.plot(history.history["val_loss"], "g-", label="Validation")
plt.title("Courbe de loss YOLO")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 12. Chargement du meilleur modele


In [ ]:
model.load_weights(str(BEST_WEIGHTS_PATH))
print("Meilleurs poids charges.")


## 13. Metriques : Precision, Recall, F1, mIoU


In [ ]:
def iou(b1, b2):
    x1 = max(b1[0] - b1[2] / 2, b2[0] - b2[2] / 2)
    y1 = max(b1[1] - b1[3] / 2, b2[1] - b2[3] / 2)
    x2 = min(b1[0] + b1[2] / 2, b2[0] + b2[2] / 2)
    y2 = min(b1[1] + b1[3] / 2, b2[1] + b2[3] / 2)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = b1[2] * b1[3] + b2[2] * b2[3] - inter
    return inter / union if union > 0 else 0.0


def compute_metrics(y_true_list, y_pred_list, iou_thres=0.5):
    TP = [0] * NB_CLASSES
    FP = [0] * NB_CLASSES
    FN = [0] * NB_CLASSES
    iou_sum = [0.0] * NB_CLASSES
    iou_cnt = [0] * NB_CLASSES

    for gt_b, pd_b in zip(y_true_list, y_pred_list):
        matched = set()
        for p in pd_b:
            pc = int(p[4])
            best, bj = 0, -1
            for j, g in enumerate(gt_b):
                if j in matched or int(g[4]) != pc:
                    continue
                v = iou(g[:4], p[:4])
                if v > best:
                    best, bj = v, j
            if best >= iou_thres and bj >= 0:
                TP[pc] += 1
                matched.add(bj)
                iou_sum[pc] += best
                iou_cnt[pc] += 1
            else:
                FP[pc] += 1
        for j, g in enumerate(gt_b):
            if j not in matched:
                FN[int(g[4])] += 1

    res = []
    for i in range(NB_CLASSES):
        P = TP[i] / (TP[i] + FP[i]) if TP[i] + FP[i] > 0 else 0
        R = TP[i] / (TP[i] + FN[i]) if TP[i] + FN[i] > 0 else 0
        F1 = 2 * P * R / (P + R) if P + R > 0 else 0
        mi = iou_sum[i] / iou_cnt[i] if iou_cnt[i] > 0 else 0
        res.append({"Precision": P, "Recall": R, "F1": F1, "mIoU": mi})

    acc = 100 * sum(TP) / max(sum(TP) + sum(FP), 1)
    miou = np.mean([r["mIoU"] for r in res if r["mIoU"] > 0]) if any(r["mIoU"] > 0 for r in res) else 0
    return res, acc, miou


def afficher_metriques(res, acc, miou):
    print(f"\nPrecision globale : {acc:.1f}%  -  mIoU : {miou:.3f}")
    print(f"{'Classe':15s} | {'Precision':9s} | {'Recall':7s} | {'F1':7s} | {'mIoU':7s}")
    print("-" * 58)
    for i, r in enumerate(res):
        print(f"{CLASS_LABELS[i]:15s} | {r['Precision']:9.3f} | {r['Recall']:7.3f} | {r['F1']:7.3f} | {r['mIoU']:7.3f}")


print("Fonctions metriques definies.")


## 14. Evaluation sur le jeu de test


In [ ]:
y_pred_yolo = model.predict(x_test, verbose=1)
y_pred_list = [
    apply_classwise_nms(boxes, iou_threshold=IOU_THRESHOLD)
    for boxes in get_box_from_yolo(y_pred_yolo, mode="pred", confidence_threshold=CONFIDENCE_THRESHOLD)
]

res, acc, miou = compute_metrics(y_test_raw, y_pred_list, iou_thres=IOU_THRESHOLD)
afficher_metriques(res, acc, miou)


## 15. Visualisation qualitative


In [ ]:
def visualiser(x, y_true, y_pred, indices=None, nb=4):
    if indices is None:
        indices = np.random.choice(len(x), min(nb, len(x)), replace=False)
    for idx in indices:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        for ax, boxes, title in zip(axes, [y_true[idx], y_pred[idx]], ["Verite terrain", "Prediction"]):
            ax.imshow(x[idx])
            for b in boxes:
                cx, cy, w, h, cid = b[0], b[1], b[2], b[3], int(b[4])
                conf = f" {b[5]:.2f}" if len(b) > 5 else ""
                rect = patches.Rectangle(
                    ((cx - w / 2) * IMAGE_W, (cy - h / 2) * IMAGE_H),
                    w * IMAGE_W,
                    h * IMAGE_H,
                    linewidth=2,
                    edgecolor=COLORS[cid],
                    facecolor="none",
                )
                ax.add_patch(rect)
                ax.text(
                    (cx - w / 2) * IMAGE_W,
                    max((cy - h / 2) * IMAGE_H - 3, 0),
                    CLASS_LABELS[cid] + conf,
                    color=COLORS[cid],
                    fontsize=8,
                    fontweight="bold",
                    bbox=dict(facecolor="black", alpha=0.4, pad=1),
                )
            ax.set_title(f"{title} - Image {idx}")
            ax.axis("off")
        plt.tight_layout()
        plt.show()


print("=== Exemples aleatoires ===")
visualiser(x_test, y_test_raw, y_pred_list, nb=6)

bons = [
    i for i, (g, p) in enumerate(zip(y_test_raw, y_pred_list))
    if any(int(gt[4]) == int(pd[4]) and iou(gt[:4], pd[:4]) >= IOU_THRESHOLD for gt in g for pd in p)
]

mauvais = [
    i for i, (g, p) in enumerate(zip(y_test_raw, y_pred_list))
    if g and any(
        not any(int(pd[4]) == int(gt[4]) and iou(gt[:4], pd[:4]) >= IOU_THRESHOLD for pd in p)
        for gt in g
    )
]

print(f"Bons exemples : {len(bons)}  -  Mauvais exemples : {len(mauvais)}")
print("\n=== Bons exemples ===")
visualiser(x_test, y_test_raw, y_pred_list, indices=bons[:4])
print("\n=== Mauvais exemples ===")
visualiser(x_test, y_test_raw, y_pred_list, indices=mauvais[:4])


## 16. Prediction sur un exemple aleatoire


In [ ]:
def showcase_random_prediction(x, y_true, y_pred):
    idx = random.randrange(len(x))
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, boxes, title in zip(axes, [y_true[idx], y_pred[idx]], ["Verite terrain", "Prediction"]):
        ax.imshow(x[idx])
        for b in boxes:
            cx, cy, w, h, cid = b[0], b[1], b[2], b[3], int(b[4])
            conf = f" {b[5]:.2f}" if len(b) > 5 else ""
            rect = patches.Rectangle(
                ((cx - w / 2) * IMAGE_W, (cy - h / 2) * IMAGE_H),
                w * IMAGE_W,
                h * IMAGE_H,
                linewidth=2,
                edgecolor=COLORS[cid],
                facecolor="none",
            )
            ax.add_patch(rect)
            ax.text(
                (cx - w / 2) * IMAGE_W,
                max((cy - h / 2) * IMAGE_H - 3, 0),
                CLASS_LABELS[cid] + conf,
                color=COLORS[cid],
                fontsize=8,
                fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.4, pad=1),
            )
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


showcase_random_prediction(x_test, y_test_raw, y_pred_list)


## 17. Matrice de confusion


In [ ]:
confusion = np.zeros((NB_CLASSES, NB_CLASSES), dtype=int)
for gt_b, pd_b in zip(y_test_raw, y_pred_list):
    for p in pd_b:
        pc = int(p[4])
        best, bgc = 0, -1
        for g in gt_b:
            v = iou(g[:4], p[:4])
            if v > best:
                best, bgc = v, int(g[4])
        if best >= IOU_THRESHOLD and bgc >= 0:
            confusion[bgc, pc] += 1

plt.figure(figsize=(8, 6))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS)
plt.xlabel("Classe predite")
plt.ylabel("Classe reelle")
plt.title("Matrice de confusion - test (IoU >= 0.5)")
plt.tight_layout()
plt.show()


## 18. Recapitulatif train / valid / test


In [ ]:
for name, xs, ys in [("Train", x_train, y_train_raw), ("Validation", x_val, y_val_raw), ("Test", x_test, y_test_raw)]:
    yp = [
        apply_classwise_nms(boxes, iou_threshold=IOU_THRESHOLD)
        for boxes in get_box_from_yolo(model.predict(xs, verbose=0), mode="pred", confidence_threshold=CONFIDENCE_THRESHOLD)
    ]
    r, a, m = compute_metrics(ys, yp, iou_thres=IOU_THRESHOLD)
    print(f"\n{'=' * 50}  {name}")
    afficher_metriques(r, a, m)
